In [1]:
# 02. Tóm tắt Dữ liệu Sạch, Quy tắc Chọn Mẫu & Chia Tập Dữ Liệu (Data Split)
**Người phụ trách:** Đạt  
**Mục tiêu:**
1. Tóm tắt nhanh kết quả Trước / Sau cleaning từ pipeline đã có (không chạy lại quy trình làm sạch).
2. Thống kê quy tắc chọn mẫu: Số chai có giá (`price`), thiếu giá và số chai có nhãn (`points`).
3. Thực hiện chia tập `Train / Validation / Test` trên các mẫu có giá theo nhóm `title` chuẩn hóa và kiểm tra chống rò rỉ dữ liệu (không chồng nhóm).

SyntaxError: invalid syntax (554444415.py, line 2)

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# Thiết lập đường dẫn theo cấu trúc repository
RAW_PATH = "../data/raw/winemag-data-130k-v2.csv"
BASIC_FORMAT_PATH = "../data/processed/wine_basic_formatted.csv"
MISSING_HANDLED_PATH = "../data/processed/wine_missing_values_handled.csv"
CLEAN_FINAL_PATH = "../data/processed/wine_outliers_noise_handled.csv"
CLEANING_LOG_PATH = "../docs/cleaning_log.csv"

# Đọc dữ liệu gốc và các bản dữ liệu đã qua pipeline cleaning
df_raw = pd.read_csv(RAW_PATH, low_memory=False)
df_basic = pd.read_csv(BASIC_FORMAT_PATH, low_memory=False) if os.path.exists(BASIC_FORMAT_PATH) else df_raw.copy()
df_clean = pd.read_csv(CLEAN_FINAL_PATH, low_memory=False)

print(f"Số dòng dữ liệu gốc (Raw): {len(df_raw):,}")
print(f"Số dòng sau chuẩn hóa cơ bản (Basic Formatted): {len(df_basic):,}")
print(f"Số dòng dữ liệu sạch cuối cùng (Outliers/Noise Handled): {len(df_clean):,}")

if os.path.exists(CLEANING_LOG_PATH):
    df_log = pd.read_csv(CLEANING_LOG_PATH)
    print("\nNhật ký cleaning_log.csv đã có sẵn:")
    display(df_log.head(10))

In [ ]:
# 1. Tính toán số dòng và duplicate trước/sau cleaning
raw_rows = len(df_raw)
clean_rows = len(df_clean)

# Kiểm tra duplicate dựa trên (title, description) hoặc toàn bộ cột đặc trưng
dup_cols = [c for c in ['title', 'description', 'points', 'price'] if c in df_raw.columns]
raw_duplicates = df_raw.duplicated(subset=['title', 'description']).sum() if {'title', 'description'}.issubset(df_raw.columns) else df_raw.duplicated().sum()
clean_duplicates = df_clean.duplicated(subset=['title', 'description']).sum() if {'title', 'description'}.issubset(df_clean.columns) else df_clean.duplicated().sum()

# 2. Thống kê giá trị price trước và sau xử lý ngoại lệ / làm sạch
raw_price_min, raw_price_max = df_raw['price'].min(), df_raw['price'].max()
clean_price_min, clean_price_max = df_clean['price'].min(), df_clean['price'].max()

summary_cleaning_df = pd.DataFrame({
    "Chỉ số đánh giá": [
        "Tổng số dòng (Rows)",
        "Số bản ghi trùng lặp (Duplicates)",
        "Số mẫu có nhãn points hợp lệ",
        "Khoảng giá (Min - Max Price USD)",
        "Giá trung vị (Median Price USD)"
    ],
    "Trước Cleaning (Raw Data)": [
        f"{raw_rows:,}",
        f"{raw_duplicates:,}",
        f"{df_raw['points'].notna().sum():,}",
        f"${raw_price_min:.1f} - ${raw_price_max:.1f}",
        f"${df_raw['price'].median():.1f}"
    ],
    "Sau Cleaning (Cleaned Data)": [
        f"{clean_rows:,}",
        f"{clean_duplicates:,}",
        f"{df_clean['points'].notna().sum():,}",
        f"${clean_price_min:.1f} - ${clean_price_max:.1f}",
        f"${df_clean['price'].median():.1f}"
    ],
    "Quyết định xử lý": [
        f"Giữ lại {clean_rows/raw_rows*100:.2f}% dữ liệu sạch",
        "Loại bỏ hoàn toàn các review trùng lặp",
        "Giữ nguyên các mẫu có điểm đánh giá chuẩn",
        "Xử lý/cắt ngưỡng các giá trị ngoại lệ cực đoan (IQR/Capping)",
        "Đảm bảo phân phối giá ổn định cho bước chia tứ phân vị & mô hình"
    ]
})

print("=== BẢNG 1: TÓM TẮT TRƯỚC VÀ SAU CLEANING ===")
display(summary_cleaning_df)

# 3. Trích xuất 2 ví dụ sửa/xử lý giá tiêu biểu để trình bày
print("\n=== HAI TRƯỜNG HỢP XỬ LÝ / SỬA GIÁ TIÊU BIỂU ===")
if 'id' in df_raw.columns and 'id' in df_clean.columns:
    merged_price = pd.merge(
        df_raw[['id', 'title', 'price']], 
        df_clean[['id', 'price']], 
        on='id', 
        suffixes=('_truoc', '_sau')
    )
    changed_prices = merged_price[
        merged_price['price_truoc'].notna() & 
        (merged_price['price_truoc'] != merged_price['price_sau'])
    ]
    if len(changed_prices) >= 2:
        display(changed_prices.head(2))
    else:
        # Hiển thị 2 trường hợp giá cực đại (outliers) đã được xử lý/loại bỏ
        extreme_cases = df_raw.nlargest(2, 'price')[['title', 'country', 'points', 'price']].copy()
        extreme_cases['Quyết định xử lý'] = f"Giá gốc vượt ngưỡng ngoại lệ (>{clean_price_max:.0f} USD) -> Đã xử lý/cắt ngưỡng trong bản sạch"
        display(extreme_cases)
else:
    extreme_cases = df_raw.nlargest(2, 'price')[['title', 'country', 'points', 'price']].copy()
    extreme_cases['Giá sau xử lý ( Ngưỡng Max mới )'] = clean_price_max
    extreme_cases['Lý do xử lý'] = "Sửa/loại bỏ ngoại lệ giá quá cao gây nhiễu hồi quy"
    display(extreme_cases)

In [ ]:
# Chọn tập dữ liệu chuẩn trước bước điền/xóa khuyết thiếu để thống kê đúng số chai thiếu giá,
# đồng thời đối chiếu với tập sạch cuối cùng
df_ref = df_basic if df_basic['price'].isna().sum() > 0 else df_raw.drop_duplicates(subset=['title', 'description'])

total_bottles = len(df_ref)
has_price = df_ref['price'].notna().sum()
missing_price = df_ref['price'].isna().sum()
has_points = df_ref['points'].notna().sum()
missing_points = df_ref['points'].isna().sum()
valid_both = (df_ref['price'].notna() & df_ref['points'].notna()).sum()

sample_selection_table = pd.DataFrame({
    "Nhóm mẫu (Sample Group)": [
        "1. Tổng số chai (sau khử trùng lặp cơ bản)",
        "2. Số chai CÓ giá (price hợp lệ)",
        "3. Số chai THIẾU giá (price bị khuyết - NaN)",
        "4. Số chai CÓ nhãn điểm (points hợp lệ)",
        "5. Số chai THIẾU nhãn điểm (points bị khuyết)",
        "6. Mẫu đủ điều kiện (Có cả price & points ở tập sạch cuối)"
    ],
    "Số lượng chai": [
        f"{total_bottles:,}",
        f"{has_price:,}",
        f"{missing_price:,}",
        f"{has_points:,}",
        f"{missing_points:,}",
        f"{len(df_clean.dropna(subset=['price', 'points'])):,}"
    ],
    "Tỷ lệ (%)": [
        "100.00%",
        f"{has_price / total_bottles * 100:.2f}%",
        f"{missing_price / total_bottles * 100:.2f}%",
        f"{has_points / total_bottles * 100:.2f}%",
        f"{missing_points / total_bottles * 100:.2f}%",
        f"{len(df_clean.dropna(subset=['price', 'points'])) / total_bottles * 100:.2f}%"
    ],
    "Quy tắc chọn mẫu": [
        "Tập nền tảng dùng để kiểm toán",
        "Giữ lại để phân tích giá–điểm & chia tập mô hình",
        "Loại khỏi bài toán hồi quy theo giá (không tự suy diễn giá ảo)",
        "Biến mục tiêu (Target) đầy đủ",
        "Không có mẫu khuyết nhãn mục tiêu",
        "Đưa vào chia Train / Validation / Test theo nhóm title"
    ]
})

print("=== BẢNG 2: TÓM TẮT SỐ CHAI CÓ/THIẾU GIÁ VÀ NHÃN POINTS ===")
display(sample_selection_table)


In [ ]:
# 1. Lọc các mẫu có đầy đủ price và points từ tập dữ liệu sạch
df_model = df_clean.dropna(subset=['price', 'points', 'title']).copy().reset_index(drop=True)

# 2. Hàm chuẩn hóa tên chai (normalized title)
def normalize_title(text):
    if pd.isna(text):
        return "unknown_title"
    text = str(text).lower().strip()
    # Chuẩn hóa khoảng trắng và ký tự dấu câu thừa nhưng giữ nguyên tên nhà sản xuất, niên vụ
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s\(\)\-\']', '', text)
    return text.strip()

df_model['title_clean'] = df_model['title'].apply(normalize_title)

# Tạo thêm cột log_price để hỗ trợ sẵn cho Dũng chạy baseline ở bước tiếp theo
df_model['log_price'] = np.log1p(df_model['price'])

# 3. Chia Train (70%) và Temp (30%) theo nhóm title_clean
gss_outer = GroupShuffleSplit(n_splits=1, train_size=0.70, random_state=42)
train_idx, temp_idx = next(gss_outer.split(df_model, groups=df_model['title_clean']))

df_train = df_model.iloc[train_idx].copy().reset_index(drop=True)
df_temp = df_model.iloc[temp_idx].copy().reset_index(drop=True)

# 4. Chia Temp (30%) thành Validation (15%) và Test (15%) theo nhóm title_clean
gss_inner = GroupShuffleSplit(n_splits=1, train_size=0.50, random_state=42)
val_idx, test_idx = next(gss_inner.split(df_temp, groups=df_temp['title_clean']))

df_val = df_temp.iloc[val_idx].copy().reset_index(drop=True)
df_test = df_temp.iloc[test_idx].copy().reset_index(drop=True)

print("Đã hoàn tất chia tập Train / Validation / Test theo nhóm `title_clean`!")